In [ ]:
# preprocessing speaker column
import pandas as pd
import re
import os

# List of input CSV files to process
input_file_paths = [
   
]

# Function to preprocess the speaker column and apply the necessary changes

def preprocess_speaker_column(input_file_path):
    
    # Extract the year from the input file name
    
    year_match = re.search(r'(\d{4})', os.path.basename(input_file_path))
    if year_match:
        year = year_match.group(1)
    else:
        raise ValueError(f"Could not extract the year from the input file name: {input_file_path}")

    # Generate the output directory path based on the extracted year
    output_directory = os.path.join(os.path.dirname(os.path.dirname(input_file_path)), f'{year}_speaker')

    # Create the directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Set the output file paths in the new folder
    output_file_path = os.path.join(output_directory, f'{year}_speaker.csv')

    # Load the CSV into a DataFrame
    try:
        df = pd.read_csv(input_file_path)
        print(f"CSV loaded successfully for {input_file_path}. Shape of data: {df.shape}")
    except Exception as e:
        print(f"Error loading CSV for {input_file_path}: {e}")
        return

    # Initialize counters for tracking changes made to 'President' and similar terms
    president_count = 0
    changes_made = 0

    # Function to clean speaker names by removing symbols
    def clean_speaker_name(speaker_name):
        # Remove all symbols and special characters
        speaker_name = re.sub(r'[^\w\s]', '', speaker_name)
        return speaker_name.strip()

    # Function to format names with just the first letter of each word capitalized
    def format_name(name):
        return ' '.join(word.capitalize() for word in name.split())

    # Define articles in all current EU languages
    eu_articles = [
        r'\b(el|la|le|l|die|der|das|il|lo|de|het|den|det|o|a|se|den|the|den|d)\s*',  # Common EU articles
        r'\b(t|an|na|den|ha|ke|il|la)\s*',  # Maltese and Finnish articles
        r'\b(o|i|tou|tis)\s*'  # Greek articles (o, i, tou, tis)
    ]

    # Define "President," "Chairperson," and related terms across all current EU languages
    president_terms = [
        r'president\w*',  # Covers President, Presidente, Président, Präsident, etc.
        r'président\w*',  # Covers French
        r'präsident\w*',  # Covers German
        r'prasident\w*',  # Covers German without umlaut
        r'proedros\w*',   # Covers Greek Πρόεδρος
        r'prezydent\w*',  # Covers Polish Prezydent
        r'predsjednik\w*', # Covers Croatian
        r'președinte\w*',  # Covers Romanian
        r'predsednik\w*',  # Covers Slovene and Slovak
        r'prezident\w*',   # Covers Czech and Slovak
        r'elnök\w*',       # Covers Hungarian
        r'uachtarán\w*',   # Covers Irish
        r'prezidents\w*',  # Covers Latvian
        r'prezidentas\w*', # Covers Lithuanian
        r'presidente\w*',  # Covers Italian, Spanish, Portuguese
        r'predsednik\w*',  # Covers Slovene
        r'predsedajuci\w*', # Covers Slavic "Predsedajuci"
        r'predseda\w*',    # Covers Slavic "Predseda"
        r'chairperson\w*',  # English term for Chairperson
        r'chair\w*',        # General term for Chair
        r'presiding\w*',    # Term for presiding chair
        r'v\w*(orsitzender|orsitzende)',  # German: Vorsitzender, Vorsitzende (Chairperson)
        r'presidente\w* de la comisión',  # Spanish: Presidente de la Comisión
        r'voorzitter\w*',  # Dutch: Chairperson (Voorzitter)
        r'presidente\w* da comissão',  # Portuguese: Presidente da Comissão
        r'presidency',    # Covers Presidency
        r'presid\w*',
        r'proedria',
        r'predrag',
        r'Przewodniczaca',
        r'Przewodniczacy',
        r'przewodnictwo',
        r'PREDSEDNICTVI',
        r'predsedkyne',
        r'VORSITZ',
        r'puhemies',
        r'Talmannen',
        r'Elnok',
        r'Presedintele',
        r'Presedinte',
        r'Presedi',
        # Covers other forms like "Presidencia"
    ]

    # Build a comprehensive regex pattern for articles followed by any president or chair-related term
    president_pattern = r'(' + '|'.join(eu_articles) + ')?(' + '|'.join(president_terms) + r')'

    # Function to normalize all variations of "President", "Chairperson," etc.
    def normalize_president(speaker_name):
        nonlocal president_count, changes_made

        # Remove articles and president-related terms followed by a name
        match = re.search(president_pattern, speaker_name, re.IGNORECASE)
        if match:
            # Check if a name follows the president term
            following_text = speaker_name[match.end():].strip()
            if following_text:
                # If a name follows, remove the president term and format the remaining name
                formatted_name = format_name(following_text)
                president_count += 1
                changes_made += 1
                return formatted_name
            else:
                # If no name follows, treat it as 'President' or 'Chairperson'
                president_count += 1
                if speaker_name.lower() != 'president':
                    changes_made += 1
                return 'President'
        return speaker_name

    # Apply the cleaning functions to the speaker column
    print(f"Processing the speaker column for {input_file_path}...")
    df['Speaker'] = df['Speaker'].apply(lambda x: normalize_president(clean_speaker_name(str(x))))

    # Ensure Party and Country columns are empty when the speaker is 'President' or similar
    df.loc[df['Speaker'] == 'President', ['Party', 'Country']] = None

    # Find all rows where Speaker has only one word and is not "President"
    one_word_speakers = df[df['Speaker'].apply(lambda x: len(str(x).split()) == 1 and x.lower() != 'president')]

    # Print the list of one-word speakers
  #  if not one_word_speakers.empty:
        #print("List of one-word speakers (excluding 'President'):")
       # print(one_word_speakers['Speaker'].tolist())
  #  else:
   #     print("No one-word speakers found (excluding 'President').")

    # Save the processed DataFrame to a new CSV
    try:
        df.to_csv(output_file_path, index=False)
        print(f"Preprocessed speaker column saved to {output_file_path}")
    except Exception as e:
        print(f"Error saving CSV for {input_file_path}: {e}")
        return

    # Print the number of President rows identified and changes made
    print(f"Number of 'President' or related rows identified: {president_count}")
    print(f"Number of changes made to normalize 'President' or related terms: {changes_made}")

# Loop through the list of input files and process each one
for input_file_path in input_file_paths:
    preprocess_speaker_column(input_file_path)


In [ ]:
#Preprocessing party column
import pandas as pd
import re
import os
from collections import Counter

# List of input CSV files to process
input_file_paths = [
   
]

# Define stop words (you can customize this list as needed)
stop_words = ["THE EARL OF", "FTA", "STOPWORD3"]  # Placeholder for actual stop words

# List of acronyms to move to 'Other Institution' column
institution_acronyms = ['CNS', 'EEC', 'CE', 'EC', 'BST', 'UNRWA', 'IGC', 'EMI']

# Hierarchical mapping of standardized party acronyms and their variations
party_variations = {
    "PES": ["PES", "PSE", "PPS", "PST", "PS", "PSSE", "SD"],
    "EPP": ["EPP", "PPE", "EVP", "EPPED", "PP", "PPEDE", "PPEOF","PPEDE"],
    "ELDR": ["ELDR", "ElDR", "LDR", "EDLR"],
    "GUE/NGL": ["GUENGL", "EULNGL", "PE", "GUE NGL", "NGLGUE", "GUEENGL","EUL-NGL","THE LEFT","GUE-NGL"],
    "Greens/EFA": ["V", "G", "GreensEFA", "GreensALE", "VertsALE", "GEFA"],
    "UEN": ["UPE", "UEN", "EN", "IEDN", "IEN", "EDN", "I EDN", "IEND", "IED", "IPE", "NELLO"],
    "RDE": ["RDE", "EDA"],
    "ARE": ["ERA"],
    "FE": ["FE"],
    "NI": ["NI", "NL"],
    "TDI": ["TDI", "TGI"],
    "EDD": ["EDD"],
    "ALDE": ["ALDE"],
    #"IND/DEM": ["INDDEM", "IND DEM","IND/DEM"],
    "ITS": ["ITS"],
    "EFD": ["EFD"],
    "ENF": ["ENF", "ENL"],
    "ECR": ["ECR", "ON BEHALF OF THE ECR GROUP","INDDEM", "IND DEM","IND/DEM"],
    "ID": ["ID"],
    "Renew": ["Renew"],
    "IEND": ["I EDN"],
    
}

# Create a reverse mapping: Each variation maps to its standardized acronym
party_mapping = {variation.upper(): standard for standard, variations in party_variations.items() for variation in variations}

# Function to clean party names by removing symbols (except for hyphens between letters)
def clean_party_name(party_name):
    # Remove all symbols except for hyphens between letters
    cleaned_name = re.sub(r'[^\w\s\-]', '', party_name)  # Remove non-word characters
    cleaned_name = re.sub(r'(?<!\w)-|-(?!\w)', '', cleaned_name)  # Remove hyphens not between letters
    
    # Remove any party names that contain numbers
    if re.search(r'\d', cleaned_name):  # Check if any numbers exist in the party name
        return ''  # Return an empty string if numbers are found
    return cleaned_name.strip().upper()  # Make sure all party names are uppercase for consistency

# Function to remove stop words from the Party column
def remove_stop_words(party_name, stop_words):
    # Remove any stop phrases (multi-word and single-word)
    for stop_word in stop_words:
        pattern = r'\b{}\b'.format(re.escape(stop_word))  # Match the exact phrase
        party_name = re.sub(pattern, '', party_name, flags=re.IGNORECASE)
    return party_name.strip()

# Function to map the party acronyms to the standardized group acronyms
def map_party_acronyms(party_name, mapping):
    # Ensure the party name is in uppercase before mapping
    party_name = party_name.upper().strip()
    return mapping.get(party_name, party_name)  # Return mapped name or original if no match

# Function to move data to the 'Other Institution' column if the party is in the institution list
def handle_institution_acronyms(row, institution_acronyms):
    party_value = row['Party'].upper().strip()
    if party_value in institution_acronyms:
        row['Other institution'] = party_value  # Move to Other Institution column
        row['Party'] = ''  # Clear the Party column
    return row

# Function to preprocess the party column and print frequency counts
def preprocess_party_column(input_file_path, stop_words, party_mapping, institution_acronyms):
    # Extract the year from the input file name
    year_match = re.search(r'(\d{4})', os.path.basename(input_file_path))
    if year_match:
        year = year_match.group(1)
    else:
        raise ValueError(f"Could not extract the year from the input file name: {input_file_path}")

    # Generate the output directory path based on the extracted year
    output_directory = os.path.join(os.path.dirname(os.path.dirname(input_file_path)), f'{year}_party')

    # Create the directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Set the output file path in the new folder
    output_file_path = os.path.join(output_directory, f'{year}_party.csv')

    # Load the CSV into a DataFrame
    try:
        df = pd.read_csv(input_file_path)
        print(f"CSV loaded successfully for {input_file_path}. Shape of data: {df.shape}")
    except Exception as e:
        print(f"Error loading CSV for {input_file_path}: {e}")
        return

    # Clean the Party column by removing symbols and stop words
    df['Party'] = df['Party'].apply(lambda x: remove_stop_words(clean_party_name(str(x)), stop_words))
    
    # Map the party acronyms to the standardized group acronyms
    df['Party'] = df['Party'].apply(lambda x: map_party_acronyms(x, party_mapping))

    # Move party data to 'Other institution' column if it matches institution acronyms
    df = df.apply(lambda row: handle_institution_acronyms(row, institution_acronyms), axis=1)

    # Print frequency of each party
    party_counts = Counter(df['Party'])
    print(f"Party frequencies for {year}:")
    for party, count in party_counts.items():
        print(f"{party}: {count}")

    # Save the processed DataFrame to a new CSV
    try:
        df.to_csv(output_file_path, index=False)
        print(f"Preprocessed party column saved to {output_file_path}")
    except Exception as e:
        print(f"Error saving CSV for {input_file_path}: {e}")
        return

# Loop through the list of input files and process each one
for input_file_path in input_file_paths:
    preprocess_party_column(input_file_path, stop_words, party_mapping, institution_acronyms)


In [ ]:
#pre processing country column
import pandas as pd
import re
import os
from collections import Counter

# List of input CSV files to process
input_file_paths = [
  
]

# Variations of country acronyms across different EU languages mapped to standardized 2-letter codes
country_variations = {
    "AT": ["AT", "AUT"],
    "BE": ["BE", "BEL"],
    "BG": ["BG", "BGR"],
    "HR": ["HR", "HRV"],
    "CY": ["CY", "CYP"],
    "CZ": ["CZ", "CZE", "CS"],
    "DK": ["DK", "DNK", "DA"],
    "EE": ["EE", "EST","ET"],
    "FI": ["FI", "FIN"],
    "FR": ["FR", "FRA"],
    "DE": ["DE", "DEU"],
    "EL": ["EL", "GRC", "GR"],  # Greece in both EU and international codes
    "HU": ["HU", "HUN"],
    "IE": ["IE", "IRL"],
    "IT": ["IT", "ITA"],
    "LV": ["LV", "LVA"],
    "LT": ["LT", "LTU"],
    "LU": ["LU", "LUX"],
    "MT": ["MT", "MLT"],
    "NL": ["NL", "NLD"],
    "PL": ["PL", "POL"],
    "PT": ["PT", "PRT"],
    "RO": ["RO", "ROU"],
    "SK": ["SK", "SVK"],
    "SI": ["SI", "SVN","SL"],
    "ES": ["ES", "ESP"],
    "SE": ["SE", "SWE", "SV"],
    "UK": ["UK", "EN","GA"],
}

# List of words or phrases to remove
words_to_remove = ["THE", "AND", "OF", "UNITED"]  # Example list of words you want to remove

# Create a reverse mapping where each variation maps to its standardized 2-letter code
country_mapping = {variation.upper(): standard for standard, variations in country_variations.items() for variation in variations}

# Function to clean country names (if there are any symbols or spaces)
def clean_country_name(country_name):
    cleaned_name = re.sub(r'[^\w\s\-]', '', country_name)  # Remove non-word characters
    cleaned_name = cleaned_name.strip().upper()  # Make uppercase and remove surrounding spaces
    return cleaned_name

# Function to remove specific words from the country name
def remove_words(country_name, words_to_remove):
    # Loop through each word/phrase in words_to_remove and remove it from the country name
    for word in words_to_remove:
        pattern = r'\b{}\b'.format(re.escape(word.upper()))  # Use word boundaries to match the exact word
        country_name = re.sub(pattern, '', country_name)
    return country_name.strip()

# Function to map country acronyms to standardized acronyms
def map_country_acronyms(country_name, mapping):
    country_name = country_name.strip().upper()
    return mapping.get(country_name, country_name)  # Return mapped acronym or original if no match

# Function to filter out invalid data from the Country column
def filter_invalid_country(country_name):
    # Remove rows with numbers or more than 3 letters or more than 1 word
    if re.search(r'\d', country_name):  # Contains numbers
        return ""
    if len(country_name) > 3:  # More than 3 letters
        return ""
    if len(country_name.split()) > 1:  # More than 1 word
        return ""
    return country_name

# Function to preprocess the country column and print frequency counts
def preprocess_country_column(input_file_path, country_mapping, words_to_remove):
    # Extract the year from the input file name
    year_match = re.search(r'(\d{4})', os.path.basename(input_file_path))
    if year_match:
        year = year_match.group(1)
    else:
        raise ValueError(f"Could not extract the year from the input file name: {input_file_path}")

    # Generate the output directory path based on the extracted year
    output_directory = os.path.join(os.path.dirname(os.path.dirname(input_file_path)), f'{year}_country')

    # Create the directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Set the output file path in the new folder
    output_file_path = os.path.join(output_directory, f'{year}_country.csv')

    # Load the CSV into a DataFrame
    try:
        df = pd.read_csv(input_file_path)
        print(f"CSV loaded successfully for {input_file_path}. Shape of data: {df.shape}")
    except Exception as e:
        print(f"Error loading CSV for {input_file_path}: {e}")
        return

    # Clean the Country column
    df['Country'] = df['Country'].apply(lambda x: clean_country_name(str(x)))
    
    # Remove specified words from the Country column
    df['Country'] = df['Country'].apply(lambda x: remove_words(str(x), words_to_remove))

    # Map the country acronyms
    df['Country'] = df['Country'].apply(lambda x: map_country_acronyms(x, country_mapping))

    # Filter out invalid country data
    df['Country'] = df['Country'].apply(lambda x: filter_invalid_country(x))

    # Print frequency of each country
    country_counts = Counter(df['Country'])
    print(f"Country frequencies for {year}:")
    for country, count in country_counts.items():
        print(f"{country}: {count}")

    # Save the processed DataFrame to a new CSV
    try:
        df.to_csv(output_file_path, index=False)
        print(f"Preprocessed country column saved to {output_file_path}")
    except Exception as e:
        print(f"Error saving CSV for {input_file_path}: {e}")
        return

# Loop through the list of input files and process each one
for input_file_path in input_file_paths:
    preprocess_country_column(input_file_path, country_mapping, words_to_remove)


In [ ]:
#1996-99 Removing empty rows and error rows
import pandas as pd
import os

def process_csv_files_in_place(directory_path):
    # Iterate through all files in the directory
    for file_name in os.listdir(directory_path):
        file_path = os.path.join(directory_path, file_name)
        
        # Check if it's a CSV file
        if not file_name.endswith(".csv"):
            continue

        # Load the CSV file
        df = pd.read_csv(file_path)

        # Track changes made
        changes = {"empty_rows_removed": 0, "rows_with_no_date_removed": 0, "NaN_cells_replaced": 0}

        # Remove completely empty rows
        initial_row_count = len(df)
        df.dropna(how="all", inplace=True)
        changes["empty_rows_removed"] = initial_row_count - len(df)

        # Remove rows where 'Date' column is empty (handle both 'Date' and 'date' cases)
        initial_row_count = len(df)
        if "Date" in df.columns:
            df = df[df["Date"].notna()]
        elif "date" in df.columns:
            df = df[df["date"].notna()]
        changes["rows_with_no_date_removed"] = initial_row_count - len(df)

        # Remove 'Form of Speech' column if it exists
        if "Form of Speech" in df.columns:
            df.drop(columns=["Form of Speech"], inplace=True)

        # Remove all columns after 'Annex_Quote' if it exists
        if "Annex_Quote" in df.columns:
            annex_quote_index = df.columns.get_loc("Annex_Quote")
            df = df.iloc[:, :annex_quote_index + 1]

        # Replace any 'NAN' entries with empty cells
        nan_cells_replaced = df.replace("NAN", "", inplace=True)
        changes["NaN_cells_replaced"] = nan_cells_replaced

        # Overwrite the original file with the cleaned data
        df.to_csv(file_path, index=False)

        # Print the summary of changes for each file
        print(f"Processing Summary for {file_name}:")
        for change, count in changes.items():
            print(f"  {change}: {count}")
        print(f"  File overwritten: {file_path}\n")

# Example usage
directory_path = 
process_csv_files_in_place(directory_path)


In [ ]:
#1999-2024 removing empty rows and error rows
import pandas as pd
import os
import re

def process_and_rename_files(directory_path):
    for file_name in os.listdir(directory_path):
        file_path = os.path.join(directory_path, file_name)

        # Process only CSV files
        if not file_name.endswith(".csv"):
            continue

        # Extract the year from the original file name using regex
        match = re.search(r'(\d{4})', file_name)
        if not match:
            print(f"Skipping file '{file_name}': Year not found in file name.")
            continue
        year = match.group(1)

        # Load the CSV file
        df = pd.read_csv(file_path)

        # Track the initial number of columns
        initial_columns = len(df.columns)

        # Ensure the 'date' column exists
        if "date" not in df.columns:
            print(f"Skipping file '{file_name}': No 'date' column found.")
            continue

        # Find the index of the 'date' column
        date_index = df.columns.get_loc("date")

        # Add the new columns right after the 'date' column
        new_columns = ["Subject", "Author", "Questions and Answers"]
        for i, col_name in enumerate(new_columns):
            df.insert(date_index + 1 + i, col_name, "")

        # Remove all columns to the right of 'Questions and Answers'
        qna_index = df.columns.get_loc("Questions and Answers")
        df = df.iloc[:, :qna_index + 1]

        # Track the number of columns removed
        columns_removed = initial_columns - len(df.columns)

        # Save the updated file with the new name
        new_file_name = f"{year}_preprocessed.csv"
        new_file_path = os.path.join(directory_path, new_file_name)
        df.to_csv(new_file_path, index=False)

        # Delete the original file if renaming is successful
        if os.path.exists(new_file_path):
            os.remove(file_path)

        # Log the summary
        print(f"Processed file: {file_name}")
        print(f"  Renamed to: {new_file_name}")
        print(f"  Columns removed: {columns_removed}")
        print(f"  Added columns: {', '.join(new_columns)}\n")

# Example usage
directory_path = 
process_and_rename_files(directory_path)
